# Semantic Search and pgvector



## Objective

- Undestand  Semantic search
- How to generate and store vector embeddings in Amazon Aurora PostgreSQL database wih pgvector extension
- Semantic search with and without index (kNN and ANN search)

## Semantic search


Semantic search is a type of search technique that aims to understand the intent and context of a user's query, rather than simply matching keywords or phrases. It goes beyond traditional keyword-based search by considering the meaning of words, the relationships between them, and the overall context of the query to deliver more relevant search results. Semantic search is important because it can help users to find the information they are looking for more quickly and easily.

Here are some examples of how semantic search is used today:
- Amazon uses semantic search to help customers find the products they are looking for. For example, if you search for "blue running shoes," Amazon will return results for shoes that are both blue and designed for running, even if you didn't use both of those keywords in your query.
- Netflix uses semantic search to recommend movies and TV shows to its users. For example, if you watch a lot of documentaries, Netflix will recommend other documentaries that you may be interested in.
- Google uses semantic search to improve the relevance of its search results. For example, if you search for "capital of France," Google will return results for Paris, even though you didn't explicitly mention Paris in your query.

One of the core components of searching textually similar items is a fixed length sentence/word embedding i.e. a  “feature vector” that corresponds to that text. The reference word/sentence embedding typically are generated offline and must be stored so they can be efficiently searched. In this use case we are using a pretrained SentenceTransformer model `amazon.titan-embed-g1-text-02` from [Amazon Titan](https://aws.amazon.com/bedrock/titan/)
 
To enable efficient searches for textually similar items, we'll use [Amazon Bedrock](https://aws.amazon.com/bedrock/) to generate fixed length sentence embeddings i.e “feature vectors” and use the Nearest Neighbor search in Amazon Aurora for PostgreSQL using the extension [pgvector](https://github.com/pgvector/pgvector). The PostgreSQL `pgvector` extension lets you store and search for points in vector space and find the "nearest neighbors" for those points. Use cases include recommendations (for example, an "other songs you might like" feature in a music application), image recognition, and fraud detection.

## pgvector

`pgvector` is an open-source extension for PostgreSQL that allows you to store and search vector embeddings for exact and approximate nearest neighbors. It is designed to work seamlessly with other PostgreSQL features, including indexing and querying.

One of the key benefits of using pgvector is that it allows you to perform similarity searches on large datasets quickly and efficiently. This is particularly useful in industries like e-commerce, where businesses need to be able to quickly search through large product catalogs to find the items that best match a customer's preferences. It supports exact and approximate nearest neighbor search, L2 distance, inner product, and cosine distance.

To further optimize your searches, you can also use pgvector's indexing features. By creating indexes on your vector data, you can speed up your searches and reduce the amount of time it takes to find the nearest neighbors to a given vector.


## kNN and ANN

kNN search, or k-Nearest Neighbors search algorithm finds the k closest vectors in the document collection to this query vector. In this context, "closest" usually means the smallest distance or highest similarity according to some metric (e.g., cosine similarity, Euclidean distance).

ANN search, or Approximate Nearest Neighbor search, is an optimization of the exact k-Nearest Neighbors (kNN) search. It aims to find the approximate nearest neighbors of a query vector, trading a small amount of accuracy for significant improvements in speed and efficiency.

**Note**

To keep things simple, we have put together multiple utility functions. You do not have to learn the code in these utility functions (unless you want to :). Do run the cells with utility functions and start from cell # 1.

## 1. Generate Vector embeddings and store with pgvector

In this section, we will see how to generate vector embeddings and store it in Amazon Aurora PostgreSQL with pgvector extension.

### 1.1  Setup
Install required python libraries for the workshop.


In [ ]:
## Install the driver for PostgreSQL

!pip install "psycopg[binary]"

### 1.2 Create DB connection

* Retrieve the database credentials from secrets manager
* Setup database connection

In [ ]:
import psycopg
from psycopg2 import sql 
import boto3
import json
import sys
import time

sm_client = boto3.client('secretsmanager')

response = sm_client.get_secret_value(SecretId='apgpg-pgvector-secret')
database_secrets = json.loads(response['SecretString'])

dbhost = database_secrets['host']
dbport = database_secrets['port']
dbuser = database_secrets['username']
dbpass = database_secrets['password']

dbconn = psycopg.connect(host=dbhost, user=dbuser, password=dbpass, port=dbport, connect_timeout=10, autocommit=True)

### 1.3. Database Utility functions

In [ ]:
# Invoke a SQL statement
import time

def invoke_sql(statement, timeit=False):
    cur = dbconn.cursor()
    try:
        start_time = time.time()
        cur.execute(statement)
        total_time = int((time.time() - start_time)*1000)
        if timeit:
            print(f"Total time taken : {total_time} ms")
            print("")
        return cur
    except Exception as error:
        print("DB statement execution error !!!", error)
        sys.exit(1)

def invoke_sql_dump_rows(statement, all=True):
    cur = invoke_sql(statement)

    if all:
        rows = cur.fetchall()
        for row in rows:
            print(row)
    else:      
        row = cur.fetchone()
        print(row)      
    cur.close()
    
  
# Utility function to display the image for products
def invoke_sql_dump_rows_with_images(sql,timeit=False):
    cur = invoke_sql(sql,timeit)
    rows = cur.fetchall()
    for row in rows:
        print(row[1])
        print(row[2])
        url = row[2]
        i = Image(url=url, width=200)
        display(i)
    cur.close()
    


### 1.4 Function to  generate embeddings

This function will use Amazon Titan Text Embeddings model to generate embeddings.

![embeddings|100x100,100%](images/gen_embeddings.png)

https://docs.aws.amazon.com/code-library/latest/ug/python_3_bedrock-runtime_code_examples.html

In [ ]:
bedrock_client = boto3.client("bedrock-runtime")

# Each model supprts discrete vector dimensions. Model we are using supports vector dimension of 1024
model_id = "amazon.titan-embed-text-v2:0"
vector_dimension = 1024

def create_embedding(input_text):
    # Create the request for the model.
    native_request = {"inputText": input_text}
    
    # Convert the native request to JSON.
    request = json.dumps(native_request)
    
    # Invoke the model with the request.
    response = bedrock_client.invoke_model(modelId=model_id, body=request)
    
    # Decode the model's native response body.
    model_response = json.loads(response["body"].read())

    # Extract and print the generated embedding and the input text token count.
    embedding = model_response["embedding"]
    
    return embedding



### 1.5  Open-source extension pgvector in PostgreSQL

In this step we'll create the extension vector and check the version of pgvector version installed. In the current lab, we have installed pgvector version `0.7.0`


In [ ]:
# Invoke a SQL to get a list of installed extensions and their versions
# pg_vector extention is already installed in the PostgreSQL database.

dbconn.execute(f"CREATE EXTENSION IF NOT EXISTS vector")

sql = "SELECT  extname, extversion FROM pg_extension where extname='vector'"

invoke_sql_dump_rows(sql)

### 1.6 Create a table with attribute of type vector

In this part, we will create a sample table with the vector datatype to store the embeddings. 


In [ ]:
dbconn.execute(f"DROP TABLE IF EXISTS sample_data;")

dbconn.execute(""" CREATE TABLE sample_data (
                      _id SERIAL PRIMARY KEY,
                      text TEXT,
                      embedding vector(1024)); """)

print("Sample table created successfully")


### 1.7 Create  embeddings

These are text input examples to demonstrate how vectors are added and queried in PostgreSQL with the pgvector extension.

**NOTE:** Running this multple times will result in the row getting added multiple times !!

In [ ]:
# Test data - Food, Sports
input_data = [
    "I love baseball",
    "I love pizza",
    "my favorite sports is football",
    "my favorite food is pasta",
    "I like enchiladas",
    "I cant miss the basketball match"
]

# Generate the embedding for data and insert to the table
for text in input_data:
    embedding = create_embedding(text)
    dbconn.execute("""INSERT INTO sample_data (text, embedding) VALUES (%s, %s);""", 
                   (text, embedding))

# Dump the count of rows in the table
invoke_sql_dump_rows(f"SELECT count(*) FROM sample_data")

Now that we have the embeddings for the words stored in the Amazon Aurora PostgreSQL database with pgvector extension, let's perform a search to find the semantically matching response.

### 1.8 Sample  search 

In the cell below, we are creating a function for sample_search, which will take the query and converts it to embedding and find the text from the sample_data table which are nearer in terms of the meaning.


In [ ]:
def sample_search(query):

   # Generate the embedding for the query
    query_embedding = create_embedding(query)

    # Do an exact search with kNN using the L2 distance operator '<->'
    # You may try different operators, the results won't change 

    ###  <#> - (negative) inner product    
    ###  <=> - cosine distance
    ###  <+> - L1 distance (added in 0.7.0) 
    ###  <~> - Hamming distance (binary vectors, added in 0.7.0)
    ###. <%> - Jaccard distance (binary vectors, added in 0.7.0)

    # Set the value of k, this is number of records to retrive.
    k = 2

    # Check out the query to retrieve the semantically closest text from the table
    sql = f"SELECT text FROM sample_data ORDER BY embedding <-> '{query_embedding}' LIMIT {k};"
    invoke_sql_dump_rows(sql)

For instance, when we search for the **Mexican food**, the result shows all the text (stored in the sample_data table) relevant to the search criteria.

In [ ]:
sample_search("I like mexican food")

In [ ]:
sample_search("I like italian food")

In [ ]:
sample_search("I am a sports fan")

## 2. Semantic search with and without Index

In this context of Semantic search, recall and performance are crucial metrics that often exist in a delicate balance. Recall refers to the proportion of relevant results that are successfully retrieved by the search system, while performance typically encompasses factors like speed and computational efficiency. High recall is desirable as it ensures that users have access to a comprehensive set of relevant results. However, achieving high recall can sometimes come at the cost of decreased performance, as more sophisticated semantic analysis and retrieval methods may require additional processing time and resources. 

The challenge for modern semantic search systems lies in optimizing both recall and performance simultaneously. This often involves employing efficient indexing strategies to quickly understand and match the semantic content of queries with relevant documents or data points. 

In this section, we will explore the recall and performance of Semantic search by using HSNW indexing strategy.

So far we have worked with a small number of rows and indexing does not really provides us any benefit. But as the number of rows increases we will start to see the benefit of ANN algorithms. 

In order to save time, we have already created the product table and populated it with roughly 10K products. We will query this table with kNN exact search and then create an index of carrying out ANN search.

```
postgres=> \d product_table_dim_1024    
             Table "public.product_table_dim_1024"     
    Column     |     Type     | Collation | Nullable | Default     
---------------+--------------+-----------+----------+---------     
 id            | text         |           | not null |     
 text          | text         |           |          |      
 embedding     | vector(1024) |           |          |     
 product_image | text         |           |          |     
 metadata      | jsonb        |           |          |     
Indexes:
    "product_table_6_dim_1024_pkey" PRIMARY KEY, btree (id)    
```
    

### 2.1 Setup search query

We will create a function `product_search` which will take the customer query, convert the query into embeddings and identify list of relevant products based on the description. 

In [ ]:
from IPython.display import Image, display
from urllib.request import urlopen


def product_search(query):
    
    query_embedding = create_embedding(query)
    k = 5
    
    # cosine distance operator
    sql = f"SELECT id, text, product_image FROM product_table_dim_1024 ORDER BY embedding <=> '{query_embedding}' LIMIT {k};"
    invoke_sql_dump_rows_with_images(sql,True)


### 2.2 Semantic search without index - kNN Search

kNN search does the full table scan on the products table and find the distance between the search query with the description for every row and provide the results based on the shortest distance. Since it goes for the full table scan, it will be perfect recall but the performance may not be optimal if the number of records in the table increases. 

At the end of the query, make a note of the total time it took for the query to execute. Later we will create index and rerun the query to see the improvement in the performance.

In [ ]:
product_search("I am looking for a toy that can teach language to children")

### 2.3 Create HNSW Index 

Let's create an index and try out ANN. Pay attention to the distance operator !!!

* **m**  (between 2 & 100, default = 16) : the max number of connections per layer. A larger M means a more interconnected, denser graph but one that will consume more memory and be slower to insert.

* **ef_construction** (between 4 & 1000, default = 64): the size of the dynamic candidate list for constructing the graph (64 by default). Bigger ef_construction leads to longer construction, but better index quality. ef_construction must be greater than or equal to 2 * m

After running the ANN search compare the query latency for exact vs ANN search. You will find the runtime stats after the query results.

In [ ]:
# Create the index

dbconn.execute(f"""CREATE INDEX ON product_table_dim_1024 
                   USING hnsw (embedding vector_cosine_ops) 
                   WITH (m=32, ef_construction=128)""")

dbconn.execute("ANALYZE product_table_dim_1024")

print("HSNW index created successfully")


### 2.4 Semantic search with HNSW index - ANN Search

Let's run the same product search query after creating the HSNW index and notice the time taken reduced drastically. Since this is small dataset, there will not be much of a difference in the recall and you may see the same output as above. 

In [ ]:
product_search("I am looking for a toy that can teach language to children")

Here we have provided some sample queries, what customer would typically search on a product website. You can un-comment any query or you can come-up with your own query to see the product listings in our products table.

- I am looking for a toy that can teach music to children
- I am looking for a toy that can teach crafts to children
- I am looking for a toy that can teach knitting to children
- I am looking for a toy that can teach engineering to children



In [ ]:
product_search("I am looking for a toy that can teach music to children")

## Conclusion
In this module , you have learned what is Semantic search and how to generate and store the vector embeddings.
Also we have compared performance and recall with and without indexes.

### Take aways
- Adapt this notebook to experiment with different models available through HuggingFace or Amazon Bedrock such as Anthropic Claude and AI21 Labs Jurassic models.
- Change the input dataset and experiment with your organizational data.


### References

[pgvector on GitHub](https://github.com/pgvector/pgvector)

[pgvector v0.80](https://www.postgresql.org/about/news/pgvector-080-released-2952/)

[Effect of Quantization](https://jkatz05.com/post/postgres/pgvector-scalar-binary-quantization/)